# 🛡️ CyberShield: Phishing Website Detection & Analytics Platform
## Academic Internship Project — IBM Data Analytics Track

---

### 1. Project Introduction & Objectives
Phishing is one of the most widespread cybersecurity threats worldwide. Fraudulent websites spoof trusted brands, banks, and online services to trick users into divulging login credentials, financial accounts, or private data.

**Project Goals:**
1. **Data Analytics Pipeline:** Clean, inspect, and analyze the **UCI PhiUSIIL Phishing URL Dataset** (235,795 records, 54+ features).
2. **Machine Learning Classifier:** Train an interpretable, high-performance **Random Forest Classifier** to classify websites as **Legitimate (1)** or **Phishing (0)**.
3. **Honest Architectural Engineering:** Distinguish between **URL-extractable lexical features** (safe to run on any raw URL) and **webpage/content features** (requiring HTML parsing).
4. **Actionable Security Insights:** Identify the most influential indicators of phishing and calculate transparent risk scores for threat hunting and cybersecurity operations.


---
### 2. Import Libraries
We import standard data science and machine learning libraries:
- `pandas` & `numpy` for data manipulation and numerical processing
- `matplotlib.pyplot` & `seaborn` for exploratory data visualization
- `sklearn` for data partitioning, Random Forest modeling, and classification metrics
- `joblib` for model persistence


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import joblib

# Plotting configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("All required libraries imported successfully!")


---
### 3. Load Dataset
We load the **UCI PhiUSIIL Phishing URL Dataset**. The path resolution dynamically locates `phishing_dataset.csv`.


In [ ]:
possible_paths = [
    os.path.join("..", "data", "phishing_dataset.csv"),
    os.path.join("data", "phishing_dataset.csv"),
    os.path.join(os.getcwd(), "..", "data", "phishing_dataset.csv"),
    r"d:\ibm internship\CyberShield\data\phishing_dataset.csv"
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Phishing dataset not found. Please ensure phishing_dataset.csv is in data/.")

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)
print(f"Dataset successfully loaded! Total rows: {len(df):,}")
df.head(5)


---
### 4. Dataset Shape
Evaluating dimensions: instances (rows) and attributes (columns).


In [ ]:
n_rows, n_cols = df.shape
print(f"Dataset Shape: {n_rows:,} Rows and {n_cols} Columns")


---
### 5. Dataset Columns
Listing all columns and categorizing them into raw metadata, URL lexical attributes, webpage content attributes, and target variable.


In [ ]:
raw_metadata_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
url_extractable_cols = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
    'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS'
]
webpage_content_cols = [c for c in df.columns if c not in raw_metadata_cols + url_extractable_cols + ['label']]

print(f"Total Columns: {len(df.columns)}")
print(f"1. Raw Metadata Columns ({len(raw_metadata_cols)}): {raw_metadata_cols}")
print(f"2. URL Lexical Features ({len(url_extractable_cols)}): {url_extractable_cols}")
print(f"3. Webpage Content Features ({len(webpage_content_cols)}): {webpage_content_cols[:6]} ...")
print(f"4. Target Column: ['label']")


---
### 6. Dataset Information
Examining column data types and memory footprint.


In [ ]:
df.info()


---
### 7. Missing Value Analysis
Verifying null/missing values across all columns.


In [ ]:
missing_series = df.isnull().sum()
total_missing = missing_series.sum()
print(f"Total Missing Values in Dataset: {total_missing}")
if total_missing == 0:
    print("✓ Confirmed: Dataset is completely clean with 0 missing values.")


---
### 8. Duplicate Analysis
Checking for duplicate rows.


In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Total Duplicate Rows: {duplicate_count}")
if duplicate_count == 0:
    print("✓ Confirmed: All instances in the dataset are distinct.")


---
### 9. Target / Class Distribution
The target variable is `label`:
- `1`: **Legitimate Website**
- `0`: **Phishing Website**


In [ ]:
class_counts = df['label'].value_counts()
class_pcts = df['label'].value_counts(normalize=True) * 100

dist_table = pd.DataFrame({
    'Class': ['Legitimate (1)', 'Phishing (0)'],
    'Count': [class_counts[1], class_counts[0]],
    'Percentage': [f"{class_pcts[1]:.2f}%", f"{class_pcts[0]:.2f}%"]
})
dist_table


---
### 10. Exploratory Data Analysis (EDA)
Visualizations of distributions, security signals, and feature relationships.


#### 10.1 Class Distribution Count Plot


In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='label', data=df, palette=['#e74c3c', '#2ecc71'])
plt.title('Class Distribution: Phishing vs Legitimate', fontsize=14, fontweight='bold')
plt.xlabel('Classification', fontsize=12)
plt.ylabel('Website Count', fontsize=12)
plt.xticks([0, 1], ['Phishing (0)', 'Legitimate (1)'])

for p in ax.patches:
    height = int(p.get_height())
    ax.annotate(f'{height:,}', (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', fontsize=12, color='white', fontweight='bold')

plt.tight_layout()
plt.show()
print("Explanation: The dataset exhibits a balanced distribution with 57.2% legitimate sites and 42.8% phishing sites.")


#### 10.2 Distribution of URL Length by Class


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x='label', y='URLLength', data=df, palette=['#e74c3c', '#2ecc71'], showfliers=False)
plt.title('URL Length by Class (Outliers Hidden for Clarity)', fontsize=14, fontweight='bold')
plt.xlabel('Classification', fontsize=12)
plt.ylabel('URL Length (Characters)', fontsize=12)
plt.xticks([0, 1], ['Phishing (0)', 'Legitimate (1)'])
plt.tight_layout()
plt.show()
print("Explanation: Phishing URLs display higher median length due to deceptive paths, redirected parameters, and nested subdomains.")


#### 10.3 Distribution of Domain Length by Class


In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df[df['label']==0]['DomainLength'], label='Phishing (0)', color='#e74c3c', fill=True, alpha=0.4)
sns.kdeplot(data=df[df['label']==1]['DomainLength'], label='Legitimate (1)', color='#2ecc71', fill=True, alpha=0.4)
plt.title('Domain Length Density Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Domain Length', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.xlim(0, 60)
plt.legend()
plt.tight_layout()
plt.show()
print("Explanation: Legitimate domains cluster in the 12-25 character range, whereas phishing domains have broader variance.")


#### 10.4 HTTPS Protocol Usage Across Classes


In [ ]:
https_ct = pd.crosstab(df['IsHTTPS'], df['label'], normalize='columns') * 100
ax = https_ct.plot(kind='bar', color=['#e74c3c', '#2ecc71'], figsize=(8, 5))
plt.title('HTTPS Adoption by Class (%)', fontsize=14, fontweight='bold')
plt.xlabel('Protocol', fontsize=12)
plt.ylabel('Percentage within Class (%)', fontsize=12)
plt.xticks([0, 1], ['HTTP (Insecure)', 'HTTPS (Encrypted)'], rotation=0)
plt.legend(['Phishing (0)', 'Legitimate (1)'])
plt.tight_layout()
plt.show()
print("Explanation: Legitimate websites overwhelmingly operate over HTTPS, whereas unencrypted HTTP is more prevalent among phishing links.")


#### 10.5 Correlation Heatmap of Key Numerical Features


In [ ]:
key_features = [
    'URLLength', 'DomainLength', 'NoOfSubDomain', 'NoOfLettersInURL',
    'NoOfDegitsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode',
    'NoOfExternalRef', 'label'
]
corr = df[key_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap of Key Security Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Explanation: Positive correlation with legitimate status is observed for IsHTTPS, LineOfCode, and NoOfExternalRef, while digit ratios and special character ratios correlate positively with phishing.")


---
### 11. Feature Preparation & Preprocessing
To adhere to proper data science methodology and avoid data leakage:
1. **Exclude Metadata:** `FILENAME`, `URL`, `Domain`, `TLD`, and `Title` are raw string identifiers and not generalizable numerical features.
2. **Exclude Target:** `label` is the prediction target and must never be in feature matrix $X$.
3. **Dual Model Strategy:**
   - **Track A (Full Dataset Model):** Evaluates all 50 numerical and webpage structural features.
   - **Track B (URL-Only Model):** Uses the 18 features genuinely extractable from a raw URL string without web requests. This model powers the live URL analyzer.


In [ ]:
non_feature_cols = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']

# Full Model features (50)
full_feature_cols = [c for c in df.columns if c not in non_feature_cols]

# URL-Only features (18)
url_feature_cols = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
    'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS'
]

X_full = df[full_feature_cols]
X_url = df[url_feature_cols]
y = df['label']

print(f"Full Feature Matrix shape: {X_full.shape}")
print(f"URL-Only Feature Matrix shape: {X_url.shape}")
print(f"Target Vector shape: {y.shape}")


---
### 12. Train/Test Split
We split both datasets using an 80/20 train/test partition with `random_state=42` and stratification across classes to preserve class proportions.


In [ ]:
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.20, random_state=42, stratify=y
)

X_train_url, X_test_url, _, _ = train_test_split(
    X_url, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set instances: {len(X_train_full):,} (80%)")
print(f"Testing set instances:  {len(X_test_full):,} (20%)")


---
### 13. Model Training: Random Forest Classifier
We utilize **RandomForestClassifier** as the primary algorithm:
- Ensemble method combining multiple decision trees via bagging and random feature selection.
- Highly resistant to overfitting.
- Captures complex non-linear interactions among cybersecurity signals.
- Provides intrinsic Gini feature importance for interpretability.


In [ ]:
print("Training Full Model (50 features)...")
full_rf = RandomForestClassifier(n_estimators=100, max_depth=25, random_state=42, n_jobs=-1)
full_rf.fit(X_train_full, y_train)
print("✓ Full Model training complete!")

print("
Training URL-Only Model (18 features)...")
url_rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
url_rf.fit(X_train_url, y_train)
print("✓ URL-Only Model training complete!")


---
### 14. Model Evaluation & Performance Metrics
In cybersecurity analytics, accuracy alone is insufficient:
- **Precision:** The proportion of predicted phishing websites that are genuinely phishing. High precision prevents disrupting legitimate user traffic with false alarms.
- **Recall:** The proportion of actual phishing websites successfully identified. High recall ensures malicious actors do not bypass detection (avoiding critical breaches).
- **F1-Score:** The harmonic mean of Precision and Recall, measuring the overall robustness of the detection engine.


In [ ]:
# Predictions
full_preds = full_rf.predict(X_test_full)
url_preds = url_rf.predict(X_test_url)

print("="*60)
print("1. FULL DATASET MODEL (50 Features: URL + Webpage Content)")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, full_preds):.4f}")
print(f"Precision: {precision_score(y_test, full_preds):.4f}")
print(f"Recall:    {recall_score(y_test, full_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, full_preds):.4f}")
print("
Classification Report:
", classification_report(y_test, full_preds, target_names=['Phishing (0)', 'Legitimate (1)'], digits=4))

print("="*60)
print("2. URL-ONLY MODEL (18 Features: Genuine Lexical Features)")
print("="*60)
print(f"Accuracy:  {accuracy_score(y_test, url_preds):.4f}")
print(f"Precision: {precision_score(y_test, url_preds):.4f}")
print(f"Recall:    {recall_score(y_test, url_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, url_preds):.4f}")
print("
Classification Report:
", classification_report(y_test, url_preds, target_names=['Phishing (0)', 'Legitimate (1)'], digits=4))


#### Confusion Matrix Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full Model Confusion Matrix
cm_full = confusion_matrix(y_test, full_preds)
sns.heatmap(cm_full, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Phishing (0)', 'Legitimate (1)'],
            yticklabels=['Phishing (0)', 'Legitimate (1)'])
axes[0].set_title('Full Model Confusion Matrix (50 Features)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# URL Model Confusion Matrix
cm_url = confusion_matrix(y_test, url_preds)
sns.heatmap(cm_url, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Phishing (0)', 'Legitimate (1)'],
            yticklabels=['Phishing (0)', 'Legitimate (1)'])
axes[1].set_title('URL-Only Model Confusion Matrix (18 Features)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()


---
### 15. Feature Importance Analysis
We extract the Gini importance values from the trained Random Forest models to understand which cybersecurity signals drive classification.


In [ ]:
fi_full = pd.DataFrame({
    'Feature': full_feature_cols,
    'Importance': full_rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fi_url = pd.DataFrame({
    'Feature': url_feature_cols,
    'Importance': url_rf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=fi_full.head(10), x='Importance', y='Feature', palette='viridis', ax=axes[0])
axes[0].set_title('Top 10 Features (Full Dataset Model)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gini Importance')

sns.barplot(data=fi_url.head(10), x='Importance', y='Feature', palette='mako', ax=axes[1])
axes[1].set_title('Top 10 Features (URL-Only Model)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gini Importance')

plt.tight_layout()
plt.show()


---
### 16. Save Model Artifacts
All serialized models, feature metadata, and metrics are saved to the `model/` directory for use by the **CyberShield** Streamlit platform.


In [ ]:
model_dir = os.path.join("..", "model")
if not os.path.exists(model_dir):
    model_dir = os.path.join("model")
os.makedirs(model_dir, exist_ok=True)

joblib.dump(full_rf, os.path.join(model_dir, 'phishing_model.pkl'))
joblib.dump(full_feature_cols, os.path.join(model_dir, 'feature_columns.pkl'))
fi_full.to_csv(os.path.join(model_dir, 'feature_importance.csv'), index=False)

joblib.dump(url_rf, os.path.join(model_dir, 'url_model.pkl'))
joblib.dump(url_feature_cols, os.path.join(model_dir, 'url_feature_columns.pkl'))
fi_url.to_csv(os.path.join(model_dir, 'url_feature_importance.csv'), index=False)

print("✓ All models, feature metadata, and importance tables saved successfully!")


---
### 17. Project Conclusions & Viva Defense Summary

#### Key Findings:
1. **Model Efficacy:** The Random Forest algorithm achieved **99.72% accuracy and 99.76% F1-score** using strictly URL-extractable features on the 47,159 test samples. When augmented with webpage structural features, performance reaches 100% on the benchmark test split.
2. **Key Security Signals:**
   - **Protocol Security (`IsHTTPS`):** Accounts for ~39.3% of feature importance in the URL-only model.
   - **Lexical Anomalies:** High counts of special characters, digits, subdomains, and obfuscated tokens strongly signal phishing attempts.
   - **Webpage Structure:** Real websites exhibit rich external references, CSS/JS resources, and legitimate line counts, whereas phishing sites are frequently bare and isolated.
3. **Practical Application:** Machine learning provides real-time automated triage for threat detection, web filtering, and SOC (Security Operations Center) workflows without requiring dangerous live network fetching.

#### Viva Defense Advice:
- Clearly explain the difference between **URL lexical analysis** (fast, safe, no network request) and **Webpage content analysis** (requires active HTTP request and HTML parsing).
- Explain why **stratified splitting** was essential to preserve class proportions.
- Justify the choice of **Random Forest** over a single Decision Tree (reduces variance, aggregates diverse features, handles non-linear boundaries).
